In [0]:
%run ./04_Function_Book

In [0]:
# Define dq_config: specifies null, format, datatype checks, and PK constraint for product data
# The config object guides the checks applied in subsequent cells

dq_config = {
    "table_name": "products",

    # NULL checks: All listed columns must have 0 nulls
    "null_checks": {
        "product_id": 0,
        "product_category_name": 0,
        "product_weight_g": 0,
        "product_length_cm": 0,
        "product_height_cm": 0,
        "product_width_cm": 0,
    },

    # FORMAT CHECKS
    "format_checks": {

        # REGEX checks: Column patterns, 0 tolerance for violations
        "regex": {
            "product_id": {
                "pattern": r"^[A-Za-z0-9]{12,}+$",
                "threshold": 0
            },
            "product_category_name": {
                "pattern": r"^[A-Za-z0-9_]+$",
                "threshold": 0
            }
        },

        # DATATYPE checks: Ensures expected type for each column
        "datatype_check": {
            "product_id": {
                "expected": "string",
                "threshold": 0
            },
            "product_category_name": {
                "expected": "string",
                "threshold": 0
            },
            "product_weight_g": {
                "expected": "int",
                "threshold": 0
            },
            "product_length_cm": {
                "expected": "int",
                "threshold": 0
            },
            "product_height_cm": {
                "expected": "int",
                "threshold": 0
            },
            "product_width_cm": {
                "expected": "int",
                "threshold": 0
            },            
        },
    },

    # Primary key uniqueness constraint
    "primary_key": {
        "column": "product_id",
        "threshold": 0
    }
}


In [0]:
# Function to aggregate quality metrics by applying all configured checks
# Combines results from null, format, and PK checks, tags with table name and timestamp
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def generate_metrics_df(df, config):
    all_results = []
    all_results += run_null_checks(df, config)
    all_results += run_format_checks(df, config)
    all_results += run_pk_check(df, config)
    for result in all_results:
        result["table_name"] = config["table_name"]
    metrics_df = spark.createDataFrame(Row(**r) for r in all_results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df

In [0]:
# Load cleaned silver-layer products table and apply all configured data quality checks
df_products = spark.table("retail_catalog.silver.products")
metrics_df = generate_metrics_df(df_products, dq_config)

In [0]:
# Return metrics DataFrame for aggregation and downstream quality monitoring
metrics_df

DataFrame[column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, table_name: string, run_time: timestamp]